# 4/7 - E/F. EFFECT and TRIGGERS
Passes and defensive actions from carrier transitions, passes under pressure, PPDA, the outcome taxonomy (overall and by zone), press triggers.

*Original sections: sections 10, 13, 15, 15b, 16.*  Barca (team 0) vs Atletico (team 1), first half.

**Run the notebooks in order** (each one reads what the earlier ones saved in `new_cache/pressure_analysis/state/`):

| # | notebook | what it covers |
|---|---|---|
| 1 | 01_pressure_signal.ipynb | B. The pressure signal |
| 2 | 02_where_zone_channel_line.ipynb | C. WHERE - zone, channel, defensive line, shape |
| 3 | 03_timing_and_possession.ipynb | D. WHEN - reaction speed and possession-level pressure |
| 4 | **04_passes_outcomes_triggers.ipynb** | E/F. EFFECT and TRIGGERS |
| 5 | 05_summary_and_registry.ipynb | G/H. Summary + event registry |
| 6 | 06_clips.ipynb | H. Clips |
| 7 | 07_exports.ipynb | I. Exports |

Shared setup (paths, tables, data contract, team-label orientation, helpers) lives in `pressure_common.py` - keep it in the same folder as the notebooks.

In [29]:
from pressure_common import *

In [30]:
# handed over by the earlier notebooks:
#   01_pressure_signal: frame_lane_summary
#   03_timing_and_possession: press_lookup
load_state(globals(), [
    'frame_lane_summary',
    'press_lookup',
])

loaded 2 variables from C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)\project\barca_atletico_first_half\new_cache\pressure_analysis\state


## 10. Passes under pressure — built from scratch (no pass_events cache)
Passes are detected directly from `carrier_track_id` transitions in `ball_frame_table`: a same-team handover is a completed pass, a change of team is an interception. Transitions shorter than `MIN_PASS_DISTANCE_M` are treated as a duel/stray touch and excluded entirely rather than counted as a pass attempt.

Pressure is checked over the `PRESSURE_LOOKBACK_SEC` window leading up to each release frame, using the opposing team's `under_team_pressure` flag from section 3 -- but a pass only counts as "under pressure" if that flag holds for `MIN_SUSTAINED_PRESSURE_FRAMES` *consecutive* frames within the window, not just a single flickering frame. A single-frame OR check let two defenders satisfy the threshold for 1 out of 25 frames and tag the whole pass as pressured, which saturated the signal regardless of how strict the per-defender gate in section 1 was.

In [31]:
PRESSURE_LOOKBACK_SEC = 1.0
PRESSURE_LOOKBACK_FRAMES = int(PRESSURE_LOOKBACK_SEC * FPS)
MIN_SUSTAINED_PRESSURE_FRAMES = 8   # ~0.3s @ 25fps; require a consecutive run, not a single flickering frame

press_flags = press_lookup.set_index(["frame_idx", "team"])["under_team_pressure"]

def was_under_pressure(passer_team, release_frame):
    passer_team = int(passer_team)
    opponent = 1 - passer_team
    release_frame = int(release_frame)
    consecutive = 0
    for f in range(max(0, release_frame - PRESSURE_LOOKBACK_FRAMES), release_frame + 1):
        if press_flags.get((f, opponent), False):
            consecutive += 1
            if consecutive >= MIN_SUSTAINED_PRESSURE_FRAMES:
                return True
        else:
            consecutive = 0
    return False

# --- detect passes + defensive actions from scratch, from carrier_track_id transitions ---
MIN_HOLD_FRAMES_PASS = 2       # ignore single-frame carrier flicker
MIN_PASS_DISTANCE_M = 3.0      # shorter than this -> duel/tackle, not a deliberate pass
MAX_RECEPTION_GAP_FRAMES = 50  # 2s @ FPS; longer gap -> ball likely went out of play, skip

carrier_seq = (
    ball_frame_table.dropna(subset=["carrier_track_id"])
    [["frame_idx", "carrier_track_id", "carrier_team"]]
    .sort_values("frame_idx")
    .reset_index(drop=True)
)
carrier_seq["run_id"] = (carrier_seq["carrier_track_id"] != carrier_seq["carrier_track_id"].shift()).cumsum()

carrier_runs_full = (
    carrier_seq.groupby("run_id")
    .agg(
        track_id=("carrier_track_id", "first"),
        team=("carrier_team", "first"),
        start_frame=("frame_idx", "first"),
        end_frame=("frame_idx", "last"),
        n_frames=("frame_idx", "count"),
    )
    .reset_index(drop=True)
)
carrier_runs_full = carrier_runs_full[carrier_runs_full["n_frames"] >= MIN_HOLD_FRAMES_PASS].reset_index(drop=True)

pos_lookup = player_frame_table.set_index(["frame_idx", "track_id"])[["pitch_x", "pitch_y"]]
role_lookup = player_frame_table.set_index(["frame_idx", "track_id"])["role"]

candidates = []
defensive_actions = []

for i in range(len(carrier_runs_full) - 1):
    passer = carrier_runs_full.loc[i]
    receiver = carrier_runs_full.loc[i + 1]

    gap_frames = receiver["start_frame"] - passer["end_frame"]
    if gap_frames > MAX_RECEPTION_GAP_FRAMES:
        continue  # too long a gap -- ball likely went out of play, not a resolved event

    try:
        passer_pos = pos_lookup.loc[(passer["end_frame"], passer["track_id"])]
        receiver_pos = pos_lookup.loc[(receiver["start_frame"], receiver["track_id"])]
    except KeyError:
        continue

    same_team = receiver["team"] == passer["team"]
    receiver_role = role_lookup.get((receiver["start_frame"], receiver["track_id"]), None)

    if (not same_team) and receiver_role == "goalkeeper":
        continue  # save/claim by the keeper -- not a midfield defensive action, exclude entirely

    dist_m = float(np.hypot(receiver_pos["pitch_x"] - passer_pos["pitch_x"],
                             receiver_pos["pitch_y"] - passer_pos["pitch_y"]))

    if dist_m < MIN_PASS_DISTANCE_M:
        if not same_team:
            defensive_actions.append({
                "defending_team": int(receiver["team"]),
                "attacking_team": int(passer["team"]),
                "action_type": "tackle",
                "frame": int(passer["end_frame"]),
                "x": float(passer_pos["pitch_x"]),
                "y": float(passer_pos["pitch_y"]),
            })
        continue  # same-team short touch = noise; different-team short = captured above as a tackle

    if not same_team:
        defensive_actions.append({
            "defending_team": int(receiver["team"]),
            "attacking_team": int(passer["team"]),
            "action_type": "interception",
            "frame": int(passer["end_frame"]),
            "x": float(passer_pos["pitch_x"]),
            "y": float(passer_pos["pitch_y"]),
        })

    candidates.append({
        "passer_track_id": passer["track_id"],
        "passer_team": int(passer["team"]),
        "receiver_track_id": receiver["track_id"],
        "receiver_team": int(receiver["team"]),
        "release_frame": int(passer["end_frame"]),
        "reception_frame": int(receiver["start_frame"]),
        "release_x": float(passer_pos["pitch_x"]),
        "release_y": float(passer_pos["pitch_y"]),
        "pass_distance_m": dist_m,
        "outcome": "completed" if same_team else "intercepted",
    })

passes_from_scratch = pd.DataFrame(candidates)
defensive_actions = pd.DataFrame(defensive_actions)

print(f"{len(passes_from_scratch)} pass attempts detected (from {len(carrier_runs_full)} carrier runs)")
print(passes_from_scratch["outcome"].value_counts())
print(f"\n{len(defensive_actions)} defensive actions captured (goalkeeper saves/claims excluded)")
print(defensive_actions["action_type"].value_counts())

451 pass attempts detected (from 670 carrier runs)
outcome
completed      361
intercepted     90
Name: count, dtype: int64

131 defensive actions captured (goalkeeper saves/claims excluded)
action_type
interception    90
tackle          41
Name: count, dtype: int64


In [32]:
passes_from_scratch["under_pressure"] = passes_from_scratch.apply(
    lambda row: was_under_pressure(row["passer_team"], row["release_frame"]), axis=1
)
passes_from_scratch["success"] = passes_from_scratch["outcome"] == "completed"

print("passes by team and pressure state:")
print(passes_from_scratch.groupby(["passer_team", "under_pressure"]).size())

pass_summary = (
    passes_from_scratch.groupby(["passer_team", "under_pressure"])["success"]
    .agg(["mean", "count"])
    .rename(columns={"mean": "success_rate"})
)
pass_summary["success_rate"] = (pass_summary["success_rate"] * 100).round(1)
print("\npass success rate by team, under pressure vs not:\n", pass_summary)

print("\ntotal passes attempted while under pressure, by team:")
print(passes_from_scratch[passes_from_scratch["under_pressure"]].groupby("passer_team").size())

passes by team and pressure state:
passer_team  under_pressure
0            False             210
             True               66
1            False             120
             True               55
dtype: int64

pass success rate by team, under pressure vs not:
                             success_rate  count
passer_team under_pressure                     
0           False                   85.2    210
            True                    75.8     66
1           False                   78.3    120
            True                    69.1     55

total passes attempted while under pressure, by team:
passer_team
0    66
1    55
dtype: int64


## 13. PPDA (Passes per Defensive Action)
Measures pressing intensity: how many passes a team allows its opponent to complete outside *that team's own defensive third* (i.e. excluding their own last-ditch defending near their own goal) before making a defensive action -- an interception or a tackle (the close-range team-change transitions captured above) -- in that same zone. Lower PPDA = more aggressive, higher-up pressing. Both the numerator and denominator are classified relative to the *defending* team (whichever team's PPDA is being computed), not the team that had the ball -- a defensive action near your own goal is just normal defending and shouldn't count as pressing, regardless of who had possession. Goalkeeper saves/claims are already excluded from `defensive_actions` at the source, so shot-stopping doesn't get counted as a pressing action.

In [33]:
def zone_for_team(x, team):
    if pd.isna(x):
        return None
    if team == 0:
        if x < 35: return "own_third"
        elif x < 70: return "middle_third"
        else: return "attacking_third"
    else:
        if x > 70: return "own_third"
        elif x > 35: return "middle_third"
        else: return "attacking_third"

ppda_by_team = {}
ppda_detail = {}

for team_id in [0, 1]:
    opponent = 1 - team_id

    # numerator: opponent's completed passes, classified in THIS team's own zone convention --
    # excluded only when they happen in this team's own defensive third (last-ditch territory)
    opp_completed = passes_from_scratch[
        (passes_from_scratch["outcome"] == "completed") &
        (passes_from_scratch["passer_team"] == opponent)
    ].copy()
    opp_completed["zone"] = opp_completed["release_x"].apply(lambda x: zone_for_team(x, team_id))
    passes_allowed = int((opp_completed["zone"] != "own_third").sum())

    # denominator: this team's own defensive actions, same zone convention
    own_actions = defensive_actions[defensive_actions["defending_team"] == team_id].copy()
    own_actions["zone"] = own_actions["x"].apply(lambda x: zone_for_team(x, team_id))
    actions_made = int((own_actions["zone"] != "own_third").sum())

    ppda_by_team[team_id] = passes_allowed / actions_made if actions_made > 0 else np.nan
    ppda_detail[team_id] = {"passes_allowed": passes_allowed, "actions_made": actions_made}

print("passes allowed / defensive actions made, by team (excluding each team's own defensive third):")
for team_id in [0, 1]:
    print(f"team {team_id}: {ppda_detail[team_id]}")

print("\nPPDA by team (lower = more aggressive pressing):")
for team_id, val in ppda_by_team.items():
    print(f"team {team_id}: {val:.2f}")

passes allowed / defensive actions made, by team (excluding each team's own defensive third):
team 0: {'passes_allowed': 124, 'actions_made': 56}
team 1: {'passes_allowed': 171, 'actions_made': 32}

PPDA by team (lower = more aggressive pressing):
team 0: 2.21
team 1: 5.34


## 15. Pressure outcome taxonomy
Pressure *volume* (`under_team_pressure`, PPDA, regain rate) says a press happened -- it doesn't say what it did to the buildup. This section classifies every resolved carrier transition into what actually happened to it: progressed forward anyway, forced backward, forced sideways, lost to an interception, lost to a tackle, or lost to a long ball / unresolved gap (previously dropped entirely by the `MAX_RECEPTION_GAP_FRAMES` filter in section 10). Same `under_pressure` definition as section 10 -- sustained pressure in the lookback window, not a single flickering frame.

This is the table meant for the LLM narrative layer: it turns "Barca pressed for 42% of Atletico's possession" into "Barca's press forced backward or sideways passes on X% of pressured touches, on top of the Y% won back outright" -- causal language a raw pressure-time percentage can't produce.

In [34]:
FORWARD_PROGRESS_THRESH_M = 3.0   # net advance toward the attacking goal, in the team's own attack direction
BACKWARD_THRESH_M = -3.0          # net retreat away from the attacking goal

def receiver_position(row):
    try:
        pos = pos_lookup.loc[(row["reception_frame"], row["receiver_track_id"])]
        return pos["pitch_x"], pos["pitch_y"]
    except KeyError:
        return np.nan, np.nan

recv_xy = passes_from_scratch.apply(receiver_position, axis=1, result_type="expand")
passes_from_scratch["receiver_x"] = recv_xy[0]
passes_from_scratch["receiver_y"] = recv_xy[1]

def progression_m(row):
    # positive = net advance toward the attacking goal, negative = net retreat, direction-corrected per team
    if pd.isna(row["receiver_x"]):
        return np.nan
    if row["passer_team"] == 0:
        return row["receiver_x"] - row["release_x"]
    else:
        return row["release_x"] - row["receiver_x"]

passes_from_scratch["progression_m"] = passes_from_scratch.apply(progression_m, axis=1)

def classify_pass_outcome(row):
    if row["outcome"] == "intercepted":
        return "lost_intercepted"
    if pd.isna(row["progression_m"]):
        return "unresolved"  # receiver position missing from player_frame_table at reception frame
    if row["progression_m"] >= FORWARD_PROGRESS_THRESH_M:
        return "progressed_forward"
    elif row["progression_m"] <= BACKWARD_THRESH_M:
        return "forced_backward"
    else:
        return "forced_sideways"

passes_from_scratch["pass_outcome_detail"] = passes_from_scratch.apply(classify_pass_outcome, axis=1)

print("pass outcome breakdown, under pressure vs not, by team:")
print(passes_from_scratch.groupby(["passer_team", "under_pressure", "pass_outcome_detail"]).size().unstack(fill_value=0))

pass outcome breakdown, under pressure vs not, by team:
pass_outcome_detail         forced_backward  forced_sideways  \
passer_team under_pressure                                     
0           False                        51               42   
            True                         20               10   
1           False                        33               26   
            True                         18               11   

pass_outcome_detail         lost_intercepted  progressed_forward  
passer_team under_pressure                                        
0           False                         31                  86  
            True                          16                  20  
1           False                         26                  35  
            True                          17                   9  


In [35]:
# --- tackles (already detected in section 10 as short-distance, different-team transitions,
#     but excluded from passes_from_scratch entirely -- pull them in as their own outcome) ---
tackle_outcomes = defensive_actions[defensive_actions["action_type"] == "tackle"][
    ["attacking_team", "frame"]
].rename(columns={"attacking_team": "team"}).copy()
tackle_outcomes["under_pressure"] = tackle_outcomes.apply(
    lambda r: was_under_pressure(r["team"], r["frame"]), axis=1
)
tackle_outcomes["pass_outcome_detail"] = "lost_tackled"
tackle_outcomes = tackle_outcomes.drop(columns="frame")

# --- large-gap carrier transitions (gap_frames > MAX_RECEPTION_GAP_FRAMES), previously dropped
#     silently in section 10 -- capture as their own bucket: ball retained after the gap
#     (throw-in, stoppage, loose-ball scramble the same team won back) vs lost after it ---
long_gap_records = []
for i in range(len(carrier_runs_full) - 1):
    passer = carrier_runs_full.loc[i]
    receiver = carrier_runs_full.loc[i + 1]
    gap_frames = receiver["start_frame"] - passer["end_frame"]
    if gap_frames <= MAX_RECEPTION_GAP_FRAMES:
        continue  # already handled above as a resolved pass / interception / tackle

    same_team = receiver["team"] == passer["team"]
    long_gap_records.append({
        "team": int(passer["team"]),
        "frame": int(passer["end_frame"]),
        "reception_frame": int(receiver["start_frame"]),
        "under_pressure": was_under_pressure(passer["team"], passer["end_frame"]),
        "pass_outcome_detail": "retained_after_gap" if same_team else "lost_after_long_ball",
    })

long_gap_events = pd.DataFrame(long_gap_records)
print(f"{len(long_gap_events)} large-gap transitions captured (previously dropped entirely)")
if len(long_gap_events):
    print(long_gap_events.groupby(["team", "under_pressure", "pass_outcome_detail"]).size())

160 large-gap transitions captured (previously dropped entirely)
team  under_pressure  pass_outcome_detail 
0     False           lost_after_long_ball    30
                      retained_after_gap      43
      True            lost_after_long_ball     3
                      retained_after_gap       8
1     False           lost_after_long_ball    28
                      retained_after_gap      32
      True            lost_after_long_ball     7
                      retained_after_gap       9
dtype: int64


In [36]:
# --- unified outcome table: every resolved carrier transition, one row each, ready for the LLM layer ---
FINAL_OUTCOME_ORDER = [
    "progressed_forward", "forced_sideways", "forced_backward",
    "lost_intercepted", "lost_tackled", "lost_after_long_ball",
    "retained_after_gap", "unresolved",
]

pass_outcomes = passes_from_scratch[["passer_team", "under_pressure", "pass_outcome_detail"]].rename(
    columns={"passer_team": "team"}
)

outcome_events = pd.concat([pass_outcomes, tackle_outcomes, long_gap_events], ignore_index=True)
outcome_events["pass_outcome_detail"] = pd.Categorical(
    outcome_events["pass_outcome_detail"], categories=FINAL_OUTCOME_ORDER, ordered=True
)

outcome_counts = (
    outcome_events.groupby(["team", "under_pressure", "pass_outcome_detail"], observed=False)
    .size()
    .unstack(fill_value=0)
)
print("carrier-transition outcomes by team and pressure state (counts):")
print(outcome_counts)

outcome_pct = outcome_counts.div(outcome_counts.sum(axis=1), axis=0) * 100
print("\nsame table as % within each team/pressure-state row:")
print(outcome_pct.round(1))

# headline: of the carrier actions a team made WHILE UNDER PRESSURE, what share were disrupted
# in some way (not clean forward progression) -- this is the opponent's press-effectiveness number,
# so index it by the team that got disrupted and remember the credit belongs to (1 - team)
DISRUPTIVE = ["forced_backward", "forced_sideways", "lost_intercepted", "lost_tackled", "lost_after_long_ball"]
pressured = outcome_events[outcome_events["under_pressure"] == True].copy()
pressured["disrupted"] = pressured["pass_outcome_detail"].isin(DISRUPTIVE)
disruption_rate = pressured.groupby("team")["disrupted"].mean() * 100

print("\n% of a team's own pressured carrier actions that were disrupted in some way")
print("(credit for the disruption belongs to the OPPOSING team's press):")
print(disruption_rate.round(1))

carrier-transition outcomes by team and pressure state (counts):
pass_outcome_detail  progressed_forward  forced_sideways  forced_backward  \
team under_pressure                                                         
0    False                           86               42               51   
     True                            20               10               20   
1    False                           35               26               33   
     True                             9               11               18   

pass_outcome_detail  lost_intercepted  lost_tackled  lost_after_long_ball  \
team under_pressure                                                         
0    False                         31            11                    30   
     True                          16             8                     3   
1    False                         26            15                    28   
     True                          17             7                     7   

pass_outc

## 15b. Pressure outcome taxonomy by pitch zone
Same taxonomy as section 15, split by the zone each transition happened in -- reuses `zone_for_team` from section 13 (PPDA), so the zone convention matches everywhere else in the notebook. Tells you not just how effective a team's pressure was, but *where*: winning the ball back in the attacking third is worth far more than shepherding it sideways in your own defensive third, and the flat section-15 numbers can't distinguish the two.

In [37]:
# --- rebuild the three outcome sources with a zone column attached ---

pass_outcomes_z = passes_from_scratch[["passer_team", "under_pressure", "pass_outcome_detail", "release_x"]].rename(
    columns={"passer_team": "team"}
).copy()
pass_outcomes_z["zone"] = pass_outcomes_z.apply(lambda r: zone_for_team(r["release_x"], r["team"]), axis=1)
pass_outcomes_z = pass_outcomes_z.drop(columns="release_x")

tackle_outcomes_z = defensive_actions[defensive_actions["action_type"] == "tackle"][
    ["attacking_team", "frame", "x"]
].rename(columns={"attacking_team": "team"}).copy()
tackle_outcomes_z["under_pressure"] = tackle_outcomes_z.apply(
    lambda r: was_under_pressure(r["team"], r["frame"]), axis=1
)
tackle_outcomes_z["pass_outcome_detail"] = "lost_tackled"
tackle_outcomes_z["zone"] = tackle_outcomes_z.apply(lambda r: zone_for_team(r["x"], r["team"]), axis=1)
tackle_outcomes_z = tackle_outcomes_z.drop(columns=["frame", "x"])

long_gap_records_z = []
for i in range(len(carrier_runs_full) - 1):
    passer = carrier_runs_full.loc[i]
    receiver = carrier_runs_full.loc[i + 1]
    gap_frames = receiver["start_frame"] - passer["end_frame"]
    if gap_frames <= MAX_RECEPTION_GAP_FRAMES:
        continue  # already covered by pass_outcomes_z / tackle_outcomes_z

    try:
        release_x = pos_lookup.loc[(passer["end_frame"], passer["track_id"])]["pitch_x"]
    except KeyError:
        release_x = np.nan

    same_team = receiver["team"] == passer["team"]
    long_gap_records_z.append({
        "team": int(passer["team"]),
        "under_pressure": was_under_pressure(passer["team"], passer["end_frame"]),
        "pass_outcome_detail": "retained_after_gap" if same_team else "lost_after_long_ball",
        "zone": zone_for_team(release_x, int(passer["team"])) if pd.notna(release_x) else None,
    })

long_gap_events_z = pd.DataFrame(long_gap_records_z)

outcome_events_z = pd.concat([pass_outcomes_z, tackle_outcomes_z, long_gap_events_z], ignore_index=True)
outcome_events_z["pass_outcome_detail"] = pd.Categorical(
    outcome_events_z["pass_outcome_detail"], categories=FINAL_OUTCOME_ORDER, ordered=True
)

n_missing_zone = outcome_events_z["zone"].isna().sum()
print(f"{n_missing_zone} events with unresolvable zone (missing position), dropped from the zone breakdown")
outcome_events_z = outcome_events_z.dropna(subset=["zone"])

2 events with unresolvable zone (missing position), dropped from the zone breakdown


In [38]:
# --- disruption rate and turnover rate, by team and zone, pressured actions only ---
ZONE_ORDER = ["own_third", "middle_third", "attacking_third"]

pressured_z = outcome_events_z[outcome_events_z["under_pressure"] == True].copy()
pressured_z["disrupted"] = pressured_z["pass_outcome_detail"].isin(DISRUPTIVE)
pressured_z["turned_over"] = pressured_z["pass_outcome_detail"].isin(
    ["lost_intercepted", "lost_tackled", "lost_after_long_ball"]
)

zone_summary = (
    pressured_z.groupby(["team", "zone"])
    .agg(n_events=("disrupted", "size"), disruption_rate=("disrupted", "mean"), turnover_rate=("turned_over", "mean"))
)
zone_summary["disruption_rate"] = (zone_summary["disruption_rate"] * 100).round(1)
zone_summary["turnover_rate"] = (zone_summary["turnover_rate"] * 100).round(1)
zone_summary = zone_summary.reindex(pd.MultiIndex.from_product([[0, 1], ZONE_ORDER], names=["team", "zone"]))

print("of a team's own pressured actions in each zone -- disruption rate and turnover rate")
print("(zone is relative to the team whose action this is; credit for the disruption/turnover belongs to the opponent's press):")
print(zone_summary)

of a team's own pressured actions in each zone -- disruption rate and turnover rate
(zone is relative to the team whose action this is; credit for the disruption/turnover belongs to the opponent's press):
                      n_events  disruption_rate  turnover_rate
team zone                                                     
0    own_third               6             83.3           50.0
     middle_third           51             60.8           23.5
     attacking_third        28             75.0           42.9
1    own_third              36             69.4           44.4
     middle_third           35             80.0           31.4
     attacking_third         7            100.0           57.1


## 16. Pressing triggers -- what state the receiver was in right before pressure arrived
No body-orientation model in the pipeline, so "back to goal" isn't directly measurable. These are position/motion-based proxies for the classic situations that invite a press: receiving near the touchline (options cut off on one side), receiving deep in your own third, receiving a backward-directed pass (reuses `progression_m` from section 15 -- ball was moving toward the receiver's own goal), receiving with few open passing lanes (reuses `frame_lane_summary` from section 2 -- isolated), and a slow/static first touch. For every completed pass reception, checks whether the receiving team came under opponent pressure within `PRESS_ONSET_WINDOW_SEC`, then reports how often a press actually follows each trigger vs the baseline rate.

In [39]:
PRESS_ONSET_WINDOW_SEC = 2.0
PRESS_ONSET_WINDOW_FRAMES = int(PRESS_ONSET_WINDOW_SEC * FPS)
TOUCHLINE_DIST_THRESH_M = 8.0
ISOLATED_LANES_THRESH = 2  # fewer than this many open passing lanes counts as isolated

reception_triggers = passes_from_scratch[passes_from_scratch["outcome"] == "completed"][
    ["receiver_track_id", "passer_team", "reception_frame", "receiver_x", "receiver_y", "progression_m"]
].rename(columns={"passer_team": "team"}).dropna(subset=["receiver_x", "receiver_y"]).copy()

reception_triggers["zone"] = reception_triggers.apply(lambda r: zone_for_team(r["receiver_x"], r["team"]), axis=1)
reception_triggers["dist_to_touchline_m"] = np.minimum(
    reception_triggers["receiver_y"], PITCH_WIDTH - reception_triggers["receiver_y"]
)

reception_triggers["trigger_touchline"] = reception_triggers["dist_to_touchline_m"] < TOUCHLINE_DIST_THRESH_M
reception_triggers["trigger_deep_own_third"] = reception_triggers["zone"] == "own_third"
reception_triggers["trigger_backward_pass"] = reception_triggers["progression_m"] < 0

# open passing lanes at the reception frame, reusing frame_lane_summary from section 2
lane_lookup = frame_lane_summary.set_index("frame_idx")["n_lanes_open"]
reception_triggers["n_lanes_open"] = reception_triggers["reception_frame"].map(lane_lookup)
reception_triggers["trigger_isolated"] = reception_triggers["n_lanes_open"] < ISOLATED_LANES_THRESH

print(f"{reception_triggers['n_lanes_open'].isna().sum()} receptions with no lane data at that frame (dropped from trigger_isolated only)")

0 receptions with no lane data at that frame (dropped from trigger_isolated only)


In [40]:
# --- first-touch speed: how quickly the receiver moved the ball in the moment right after control ---
FIRST_TOUCH_WINDOW_FRAMES = int(0.4 * FPS)  # ~0.4s right after the touch
STATIC_TOUCH_SPEED_MPS = 1.5

def first_touch_speed(row):
    try:
        p0 = pos_lookup.loc[(row["reception_frame"], row["receiver_track_id"])]
        p1 = pos_lookup.loc[(row["reception_frame"] + FIRST_TOUCH_WINDOW_FRAMES, row["receiver_track_id"])]
    except KeyError:
        return np.nan
    dist_m = float(np.hypot(p1["pitch_x"] - p0["pitch_x"], p1["pitch_y"] - p0["pitch_y"]))
    return dist_m / (FIRST_TOUCH_WINDOW_FRAMES / FPS)

reception_triggers["first_touch_speed_mps"] = reception_triggers.apply(first_touch_speed, axis=1)
reception_triggers["trigger_static_touch"] = reception_triggers["first_touch_speed_mps"] < STATIC_TOUCH_SPEED_MPS

print(f"{reception_triggers['first_touch_speed_mps'].isna().sum()} receptions with no first-touch speed "
      f"(receiver missing from player_frame_table {FIRST_TOUCH_WINDOW_FRAMES} frames later -- likely a pass near a stoppage)")

0 receptions with no first-touch speed (receiver missing from player_frame_table 10 frames later -- likely a pass near a stoppage)


In [41]:
# --- did opponent pressure actually follow this reception, within PRESS_ONSET_WINDOW_SEC? ---
def press_follows_reception(row):
    team = int(row["team"])
    opponent = 1 - team
    frame0 = int(row["reception_frame"])
    for f in range(frame0, frame0 + PRESS_ONSET_WINDOW_FRAMES + 1):
        if press_flags.get((f, opponent), False):
            return True
    return False

reception_triggers["press_followed"] = reception_triggers.apply(press_follows_reception, axis=1)

TRIGGER_COLS = ["trigger_touchline", "trigger_deep_own_third", "trigger_backward_pass",
                "trigger_isolated", "trigger_static_touch"]

baseline_rate = reception_triggers["press_followed"].mean() * 100
print(f"baseline: a press follows any reception {baseline_rate:.1f}% of the time (n={len(reception_triggers)})\n")

for col in TRIGGER_COLS:
    with_trigger = reception_triggers[reception_triggers[col] == True]
    without_trigger = reception_triggers[reception_triggers[col] == False]
    print(f"{col}: press followed {with_trigger['press_followed'].mean()*100:.1f}% of the time "
          f"(n={len(with_trigger)}) vs {without_trigger['press_followed'].mean()*100:.1f}% otherwise")

print("\ntrigger prevalence and press-follow rate by receiving team (%):")
print((reception_triggers.groupby("team")[TRIGGER_COLS + ["press_followed"]].mean() * 100).round(1))

baseline: a press follows any reception 46.8% of the time (n=361)

trigger_touchline: press followed 72.9% of the time (n=85) vs 38.8% otherwise
trigger_deep_own_third: press followed 31.3% of the time (n=99) vs 52.7% otherwise
trigger_backward_pass: press followed 43.6% of the time (n=163) vs 49.5% otherwise
trigger_isolated: press followed 78.3% of the time (n=23) vs 44.7% otherwise
trigger_static_touch: press followed 39.3% of the time (n=89) vs 49.3% otherwise

trigger prevalence and press-follow rate by receiving team (%):
      trigger_touchline  trigger_deep_own_third  trigger_backward_pass  \
team                                                                     
0                  20.1                    15.7                   40.6   
1                  29.5                    47.7                   53.0   

      trigger_isolated  trigger_static_touch  press_followed  
team                                                          
0                  5.7                  19.

## Hand-off
Saves what the later notebooks need.

In [42]:
save_state(globals(), [
    'PRESS_ONSET_WINDOW_FRAMES',
    'TRIGGER_COLS',
    'carrier_runs_full',
    'defensive_actions',
    'disruption_rate',
    'long_gap_events',
    'outcome_pct',
    'pass_summary',
    'passes_from_scratch',
    'ppda_by_team',
    'ppda_detail',
    'reception_triggers',
    'zone_summary',
], notebook='04_passes_outcomes_triggers.ipynb')

saved 13 variables to C:\Users\user\Desktop\Football_cv_project\football-pipeline (1)\project\barca_atletico_first_half\new_cache\pressure_analysis\state
  PRESS_ONSET_WINDOW_FRAMES         int             0.00 MB
  TRIGGER_COLS                      list            0.00 MB
  carrier_runs_full                 DataFrame       0.03 MB
  defensive_actions                 DataFrame       0.01 MB
  disruption_rate                   Series          0.00 MB
  long_gap_events                   DataFrame       0.01 MB
  outcome_pct                       DataFrame       0.00 MB
  pass_summary                      DataFrame       0.00 MB
  passes_from_scratch               DataFrame       0.05 MB
  ppda_by_team                      dict            0.00 MB
  ppda_detail                       dict            0.00 MB
  reception_triggers                DataFrame       0.03 MB
  zone_summary                      DataFrame       0.00 MB
